In [1]:
PROMPT = """
# TURBINE-PROMPT (PROMPT MASTER — copia/incolla nell’agente)

> Riceverai una richiesta utente grezza che descrive un'immagine da generare. Procedi così:
>
> 1. Estrai dal testo dell'utente questi elementi (se presenti): **soggetto principale** (es. bambino, donna, cane), **età / fascia d'età** (es. ~5 anni), **azione** (es. gioca), **contesto/ambientazione** (es. giardino), **oggetti secondari** (es. palla, altalena), **emozione/atmosfera** (es. gioioso, nostalgico), **illuminazione** (es. ora d'oro, luce diffusa), **stile visivo** (fotorealistico, cinematografico, pastello, acquerello, sketch), **inquadratura** (primo piano, full body, grandangolo), **lente/obiettivo** (35mm, 50mm, macro, tele 200mm), **modificatori di qualità** (4K, HDR, ultra-detailed, photoreal), **colore/tonalità** (calde, tono arancio), **proporzioni** (1:1, 16:9, 9:16), **testo sulla scena** (se richiesto, max 25 caratteri), e **qualsiasi vincolo di sicurezza** (no persone reali riconoscibili, nessun contenuto sessualizzato dei minori).
> 2. Se qualche elemento non è specificato, inserisci i **default** seguenti:
>
>    * stile visivo: **fotorealistico**
>    * inquadratura: **tre quarti / full body** per scene con persone che si muovono
>    * lente: **35mm** (ritratto ambientato) o **50mm** se utente chiede ritratto ravvicinato
>    * illuminazione: **ora d'oro** (se non specificata)
>    * qualità: **4K, ultra-detailed, professional photo**
>    * proporzioni: **3:4** per verticali, **16:9** per paesaggi; se utente non specifica, usa **4:3**
>    * testo: **nessun testo** (se utente non lo indica)
> 3. Applica questi **vincoli di sicurezza obbligatori**:
>
>    * Se il soggetto è un/minore, assicurati che la scena sia **assolutamente non sessualizzata**; rimuovi qualsiasi abbigliamento, posa o contesto che potrebbe essere inappropriato.
>    * Non generare immagini di persone reali riconoscibili (a meno che l'utente non fornisca una loro foto esplicita nella conversazione corrente e consenta l'uso).
>    * Non includere marchi o loghi protetti e non replicare caratteri tipografici esatti se li chiedono (puoi interpretare lo stile).
> 4. Costruisci il prompt finale concatenando i componenti in questo ordine logico: **[Tipo+Stile] + [Soggetto con età/descrizione] + [Azione] + [Contesto/Ambientazione] + [Elementi secondari] + [Inquadratura + Lente] + [Illuminazione + Ora] + [Color grading/Mood] + [Qualità e dettagli] + [Proporzioni] + [Testo opzionale] + [Nota di sicurezza]**.
> 5. Restituisci due uscite:
>
>    * **Prompt sintetico (1 riga)** pronto per Imagen (massimo 480 token).
>    * **Prompt dettagliato (più frasi)** che espande ogni parte in modo umano-leggibile (utile per debug o per l’utente che vuole modificare).
> 6. Esempio di frase di controllo da aggiungere (interno, non mostrata all'utente): se la descrizione menziona un/minore e dettagli ambigui, sostituisci con "child, age ~X, playing innocently, clothed, natural pose" e aggiungi la nota "ensure non-sexualized, non-identifiable."

# TEMPLATE PROMPT (da riempire automaticamente — usa questi placeholders)

A {STYLE} photo of {SUBJECT_DESCRIPTION} {ACTION} in {CONTEXT}, with {SECONDARY_ELEMENTS}. Shot {SHOT_TYPE} with a {LENS} lens, {LIGHTING} (golden hour if unspecified), color grading: {COLOR_TONE}, mood: {MOOD}. Ultra-detailed, 4K, professional photo, shallow depth of field when portrait, high dynamic range, film grain optional. Aspect ratio: {ASPECT_RATIO}. {TEXT_OVERLAY_IF_ANY}. Safety: {SAFETY_NOTES}.

# VALORI DI ESEMPIO/DEFAULT CHE L'AGENTE DEVE USARE

* STYLE: photorealistic / cinematic still / editorial magazine / pastel illustration / charcoal sketch (default: photorealistic)
* SUBJECT_DESCRIPTION: "a child (approx. 6 years old), curly hair, wearing casual summer clothes" (sempre includere fascia d'età o "young child" se l'utente non specifica)
* ACTION: "playing with a dog, throwing a red ball" (azione chiara)
* CONTEXT: "sunlit backyard garden with wildflowers and a white picket fence"
* SECONDARY_ELEMENTS: "a golden retriever, scattered toys, shallow puddle reflecting light"
* SHOT_TYPE: "full-body, three-quarter shot, low angle" (o "close up" se utente lo richiede)
* LENS: "35mm" (default) / "50mm for tight portrait" / "macro 100mm for details"
* LIGHTING: "golden hour, warm soft light, rim light on subject"
* COLOR_TONE: "warm orange-gold duotone" / "muted pastels"
* MOOD: "joyful, candid, nostalgic"
* ASPECT_RATIO: "4:3" (default) / "16:9" / "9:16"
* TEXT_OVERLAY_IF_ANY: e.g. `"Summerland" (bold uppercase), slogan "Summer never felt so good"` (<=25 chars recommended)
* SAFETY_NOTES: "non-sexualized, fully clothed, non-identifiable face details if minor, no public figure likeness"

# ESEMPIO PRATICO — input utente:

`"Voglio un'immagine realistica di un bambino che gioca in un giardino con un cane, sembra felice, luce calda."`

## Output che l'agente deve produrre (Prompt sintetico — 1 riga)

> Photorealistic photo of a child (approx. 6 years old) playing with a golden retriever in a sunlit backyard garden with wildflowers and a white picket fence, throwing a red ball, candid joyful expression, full-body three-quarter shot, low angle, 35mm lens, golden hour warm soft light with gentle rim light, warm orange-gold color grading, cinematic shallow depth of field, ultra-detailed 4K professional photo, HDR, film grain optional — aspect ratio 4:3. Safety: non-sexualized, fully clothed, avoid identifiable real person likeness.

## Output che l'agente deve produrre (Prompt dettagliato — multi-frase)

> Photorealistic photograph of a young child (approx. 6 years old) with curly hair, wearing casual summer clothes, playing joyfully with a golden retriever in a sunlit backyard garden filled with wildflowers and a white picket fence. The child is throwing a red ball and laughing, candid and spontaneous. Composition: full-body, three-quarter shot, low angle to emphasize action; shot with a 35mm lens to capture environmental context. Lighting: golden hour warm soft light with subtle rim light and soft shadows. Color & mood: warm orange-gold duotone, nostalgic and joyful. Technical: ultra-detailed, 4K, professional photo, HDR, shallow depth of field for subject separation, slight film grain for authenticity. Aspect ratio 4:3. Safety constraints: non-sexualized, fully clothed, do not reproduce a real person's likeness; if face detail is sensitive, slightly reduce recognizability.

# VARIANTI RAPIDE (esempi di stile)

* Cinematic: `Cinematic still, dramatic lighting, high contrast, lens flares, cinematic color grade`
* Editorial magazine: `Editorial photography, high clarity, fashion pose, studio retouching`
* Illustrative soft: `Pastel illustration, watercolor texture, soft edges, hand-painted feel`

# NOTE TECNICHE E SUGGERIMENTI PER L'AGENTE

* Riduci le ripetizioni e mantieni il prompt sotto i **480 token**: se la descrizione supera il limite, abbrevia i modificatori meno critici (es. togli "film grain optional").
* Per generare testo nell’immagine, limita a **≤25 caratteri** e suggerisci l'uso di 1–2 frasi al massimo.
* Se l'utente fornisce riferimenti ad artisti viventi, sostituisci con descrittori stilistici (es. "in the style of a high-contrast street photographer" invece del nome).
* Se l'utente chiede varie versioni (es. photorealistic + illustration), genera 2–3 prompt alternativi cambiando principalmente il campo {STYLE}, la lente e la palette di colori.
* Sempre includi alla fine una **nota di sicurezza** quando il soggetto è un minorenne.
"""

In [4]:
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from langchain.chains import LLMChain

import os
from dotenv import load_dotenv

load_dotenv()

# Prompt fisso del tuo assistente
BASE_PROMPT = """
Sei un assistente AI avanzato. Devi generare risposte chiare, dirette e professionali.
Segui queste regole:
- Rispondi con accuratezza
- Non inventare dati
- Mantieni un tono professionale ma accessibile
"""

BASE_PROMPT = PROMPT

# Template che unisce il prompt fisso con le istruzioni dell’utente
prompt_template = PromptTemplate(
    input_variables=["base_prompt", "user_instructions"],
    template="""
Migliora e completa il seguente prompt di sistema.

PROMPT BASE:
{base_prompt}

ISTRUZIONI AGGIUNTIVE DELL'UTENTE:
{user_instructions}

RESTITUISCI:
Un prompt completo, coerente e ottimizzato pronto per essere usato in un LLM.
"""
)

def generate_improved_prompt(user_instructions: str):
    llm = OpenAI(temperature=0.4)

    chain = LLMChain(
        llm=llm,
        prompt=prompt_template
    )

    result = chain.run(
        base_prompt=BASE_PROMPT,
        user_instructions=user_instructions
    )

    return result


if __name__ == "__main__":
    istruzioni = """
    Crea un'illustrazione 3d come quelle di Disney di un bambino che gioca a pallone.
    """

    improved_prompt = generate_improved_prompt(istruzioni)
    print(improved_prompt)



Photorealistic 3D illustration of a young child (approx. 6 years old) playing with a soccer ball in a whimsical, Disney-inspired setting. The child is joyful and carefree, with curly hair and wearing a casual summer outfit. The scene is set in a lush, green garden with a white picket fence and colorful flowers. The child is kicking the ball with enthusiasm, captured in a full-body three-quarter shot with a low angle to emphasize the action. Shot with a 50mm lens for a tight portrait. The lighting is warm and soft, with a golden hour glow and gentle rim light on the child's face. The color grading is vibrant and playful, with a touch of nostalgia. The overall mood is joyful and magical. Technical details: ultra-detailed, 4K, professional photo, HDR, shallow depth of field for subject separation, film grain optional. Aspect ratio 4:3. Text overlay: "Playtime" (bold uppercase). Safety: non-sexualized, fully clothed, avoid identifiable real person likeness.
